## Epic 2 — Preprocessing Pipeline

Transforms the raw `MetObjects.csv` into a clean, model-ready feature matrix.
Covers null handling, column selection, and feature engineering for both the supervised (Department classification) and unsupervised (clustering) tracks.

In [1]:
import os

import numpy as np
import pandas as pd
import scipy.sparse
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

In [2]:
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data", "MetObjects.csv")

# Setting low_memory=False forces pandas to read the entire column before deciding its dtype, which is slower but correct.
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print("Loaded:", df_raw.shape)

Loaded: (484956, 54)


In [3]:
# ── Column definitions (single source of truth) ────────────────────────────
# Source: reports/feature_engineering.md

COLS_TO_DROP = [
    # Identifiers
    "Object ID", "Object Number", "Constituent ID",
    # URLs — no feature signal
    "Link Resource", "Artist ULAN URL", "Artist Wikidata URL",
    "Object Wikidata URL", "Tags AAT URL", "Tags Wikidata URL",
    # 100% null / constant
    "Metadata Date", "Repository",
    # Geographic columns >90% null
    "River", "State", "Locus", "County", "Locale",
    "Excavation", "Subregion", "Region", "City",
    # Admin fields with no object-level signal
    "Rights and Reproduction", "Portfolio", "Gallery Number", "Credit Line",
    # Redundant / low-signal artist fields
    "Artist Prefix", "Artist Suffix", "Artist Alpha Sort",
    "Artist Display Bio", "Artist Display Name",
    # Flagged — dropping for now
    "Dynasty", "Reign", "Geography Type", "Country",
    "Artist Begin Date", "Artist End Date",
    "Dimensions", "Title", "Object Date",
]

COLS_NUMERIC = [
    "AccessionYear",          # 0.8% null → fill with median year
]

COLS_CATEGORICAL = [
    "Medium",                 # 1.5% null  — TF-IDF after comma-splitting
    "Tags",                   # 60.3% null — TF-IDF after pipe-splitting
    "Classification",         # 16.2% null — TF-IDF or top-N encode
    "Object Name",            # 0.5% null  — top-N frequency encode
    "Culture",                # 57.1% null — top-100 + "Other"
    "Period",                 # 81.2% null — top-N + "Other"
    "Artist Nationality",     # 41.7% null — top-N + "Other"
    "Artist Role",            # 41.7% null — label encode
    "Artist Gender",          # 78.0% null — binary flag
]

COLS_KEEP = [
    "Department",             # classification target — kept, encoded separately
    "Object Begin Date",      # int64, 0% null — clip + derive features
    "Object End Date",        # int64, 0% null — clip + derive features
    "Is Highlight",           # bool, 0% null
    "Is Timeline Work",       # bool, 0% null
    "Is Public Domain",       # bool, 0% null
]

print("Columns accounted for:",
      len(COLS_TO_DROP) + len(COLS_NUMERIC) + len(COLS_CATEGORICAL) + len(COLS_KEEP),
      "/ 54 total")

Columns accounted for: 54 / 54 total


In [6]:
df = df_raw.drop(columns=COLS_TO_DROP)

print(f"Shape before drop : {df_raw.shape}")
print(f"Shape after drop  : {df.shape}")
print(f"Columns removed   : {df_raw.shape[1] - df.shape[1]}")
print()
assert "Department" in df.columns, "ERROR: Department was accidentally dropped!"
print("Test - Department column intact!!")

Shape before drop : (484956, 54)
Shape after drop  : (484956, 16)
Columns removed   : 38

Test - Department column intact!!


### Missing Value Imputation

In [7]:
# Null counts before imputation — only show columns that have any nulls
null_before = df.isna().sum()
null_before = null_before[null_before > 0].sort_values(ascending=False)

print(f"Columns with nulls: {len(null_before)}")
print(f"Total null cells  : {null_before.sum():,}")
print()
display(null_before.rename("null_count").to_frame())

Columns with nulls: 10
Total null cells  : 1,838,500



,null_count
Period,393813
Artist Gender,378474
Tags,292501
Culture,276766
Artist Role,202443
Artist Nationality,202443
Classification,78717
Medium,7215
AccessionYear,3862
Object Name,2266


In [9]:
COLS_BOOL = ["Is Highlight", "Is Public Domain", "Is Timeline Work"]

# Numeric: coerce to float first (AccessionYear is stored as string in the CSV),
# then fill nulls introduced by coercion with the column median
for col in COLS_NUMERIC:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Categorical: fill with "Unknown" so the model sees a real category, not NaN
for col in COLS_CATEGORICAL:
    df[col] = df[col].fillna("Unknown")

# Booleans: fill with False (absence of a flag is the safe neutral value)
for col in COLS_BOOL:
    df[col] = df[col].fillna(False)

### Notes

- AccessionYear is the only numeric column here,
  and year values are likely slightly skewed (e.g.
   a burst of acquisitions in a particular era).
  Mean is pulled by outliers — if a handful of
  records have a weird year like 1800 or 2050, the
   mean shifts. Median is the middle value of the
  sorted column, so outliers can't distort it.
- Top-N frequency encode / label encode:
  "Unknown" becomes its own category. The model
  learns a weight for it — effectively learning "objects with no Culture info tend to land in
  these departments." That's genuinely useful
  signal, because missing = anonymous object is
  not random noise.

In [14]:
remaining_nulls = df.isna().sum().sum()
print(f"Total null cells after imputation: {remaining_nulls}")
assert remaining_nulls == 0, f"Expected 0 nulls, found {remaining_nulls}"
print("Test -- All nulls resolved!")

Total null cells after imputation: 0
Test -- All nulls resolved!


### Simple Column Encoding

In [16]:
# Boolean → int  (True=1, False=0)
COLS_BOOL = ["Is Highlight", "Is Public Domain", "Is Timeline Work"]
for col in COLS_BOOL:
    df[col] = df[col].astype(int)

# AccessionYear was coerced to float in the imputation step; cast to int now
# that nulls are gone (a year like 1978.0 should be stored as 1978)
df["AccessionYear"] = df["AccessionYear"].astype(int)

In [19]:
# Clip date outliers before deriving features
# Values below -7000 are placeholder junk (2,074 records per EDA)
df["Object Begin Date"] = df["Object Begin Date"].clip(lower=-7000, upper=2026)
df["Object End Date"]   = df["Object End Date"].clip(lower=-7000, upper=2026)

# Derive temporal features
# Reference year matches the upper clip so object_age is always >= 0
df["object_age"]  = 2026 - df["Object End Date"]
df["object_span"] = (df["Object End Date"] - df["Object Begin Date"]).clip(lower=0)

In [20]:
COLS_SIMPLE_NUMERIC = COLS_BOOL + ["AccessionYear", "object_age", "object_span"]

assert df[COLS_BOOL].isin([0, 1]).all().all(), "Boolean columns contain values other than 0/1"
assert df["object_age"].min() >= 0,  "object_age has negative values — check clipping"
assert df["object_span"].min() >= 0, "object_span has negative values — check clipping"

print(df[COLS_SIMPLE_NUMERIC].dtypes)

Is Highlight        int64
Is Public Domain    int64
Is Timeline Work    int64
AccessionYear       int64
object_age          int64
object_span         int64
dtype: object


### High-Cardinality Categorical Encoding

In [21]:
# ── Culture: frequency encoding ─────────────────────────────────────────────
# Replace each culture string with how many times it appears in the dataset.
# High-frequency cultures (e.g. "American") get large numbers;
# rare cultures get small ones — the numeric value carries popularity signal.

print(f"Culture unique values BEFORE: {df['Culture'].nunique()}")

culture_freq_map = df["Culture"].value_counts().to_dict()
df["Culture"] = df["Culture"].map(culture_freq_map).fillna(0).astype(int)

print(f"Culture unique values AFTER : {df['Culture'].nunique()}  (now frequency counts)")

Culture unique values BEFORE: 7313
Culture unique values AFTER : 218  (now frequency counts)


In [22]:
# ── Object Name: top-50 encoding ────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

print(f"Object Name unique values BEFORE: {df['Object Name'].nunique()}")

top_object_names = set(df["Object Name"].value_counts().head(50).index)
df["Object Name"] = df["Object Name"].apply(
    lambda x: x if x in top_object_names else "Other"
)

object_name_encoder = LabelEncoder()
df["Object Name"] = object_name_encoder.fit_transform(df["Object Name"])

print(f"Object Name unique values AFTER : {df['Object Name'].nunique()}  (label-encoded integers)")

Object Name unique values BEFORE: 28632
Object Name unique values AFTER : 51  (label-encoded integers)


In [23]:
# ── Classification: top-50 encoding ─────────────────────────────────────────
print(f"Classification unique values BEFORE: {df['Classification'].nunique()}")

top_classifications = set(df["Classification"].value_counts().head(50).index)
df["Classification"] = df["Classification"].apply(
    lambda x: x if x in top_classifications else "Other"
)

classification_encoder = LabelEncoder()
df["Classification"] = classification_encoder.fit_transform(df["Classification"])

print(f"Classification unique values AFTER : {df['Classification'].nunique()}  (label-encoded integers)")

Classification unique values BEFORE: 1245
Classification unique values AFTER : 51  (label-encoded integers)


### TF-IDF Encoding — Medium and Tags

In [24]:
# ── Medium: preprocessing ────────────────────────────────────────────────────
# "Bronze, patinated, gilt" → "Bronze patinated gilt" so the vectorizer
# sees three tokens, not "bronze," with a trailing comma baked in.

df["Medium"] = (
    df["Medium"]
    .fillna("")
    .str.replace(",", " ", regex=False)          # commas → spaces
    .str.replace(r"\s+", " ", regex=True)        # collapse double spaces
    .str.strip()
)

In [25]:
# ── Tags: preprocessing ──────────────────────────────────────────────────────
# "Portraits|Men|Costumes" → "Portraits Men Costumes" so each tag is one token.
# Nulls become "" so the vectorizer produces an all-zero row (not an error).

df["Tags"] = (
    df["Tags"]
    .fillna("")
    .str.replace("|", " ", regex=False)          # pipes → spaces
    .str.strip()
)

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Medium TF-IDF ─────────────────────────────────────────────────────────────
medium_vectorizer = TfidfVectorizer(
    max_features=100,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)
medium_tfidf = medium_vectorizer.fit_transform(df["Medium"])

# ── Tags TF-IDF ───────────────────────────────────────────────────────────────
tags_vectorizer = TfidfVectorizer(
    max_features=50,
    ngram_range=(1, 1),
    sublinear_tf=True,
)
tags_tfidf = tags_vectorizer.fit_transform(df["Tags"])

print(f"medium_tfidf shape : {medium_tfidf.shape}")
print(f"tags_tfidf shape   : {tags_tfidf.shape}")

medium_tfidf shape : (484956, 100)
tags_tfidf shape   : (484956, 50)


In [29]:
print("Top 20 Medium features:")
print(medium_vectorizer.get_feature_names_out()[:20])

print()
print("Top 20 Tags features:")
print(tags_vectorizer.get_feature_names_out()[:20])

Top 20 Medium features:
['albumen' 'albumen photograph' 'albumen silver' 'alloy' 'aquatint'
 'black' 'black chalk' 'black ink' 'blue' 'brass' 'bronze' 'brown'
 'brown ink' 'brush' 'canvas' 'chalk' 'color' 'color lithograph'
 'color paper' 'colored']

Top 20 Tags features:
['abstraction' 'actors' 'actresses' 'and' 'angels' 'animals'
 'architecture' 'arms' 'athletes' 'baseball' 'birds' 'boats' 'boys'
 'buildings' 'carriages' 'children' 'christ' 'coat' 'dogs' 'dragons']
